### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="forest_fires",
    dataset_year="2008",
    domain_str="environmental science & climate",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5D88D",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/162/forest+fires.zip && unzip forest+fires.zip && rm forest+fires.zip forestfires.names
mkdir -p local-data-warehouse/forest_fires && mv forestfires.csv local-data-warehouse/forest_fires/
""",
    # References
    academic_reference_bibtex=r"""@article{cortez2007data,
  title={A data mining approach to predict forest fires using meteorological data},
  author={Cortez, Paulo and Morais, An{\'\i}bal de Jesus Raimundo},
  year={2007},
  publisher={Associa{\c{c}}{\~a}o Portuguesa para a Intelig{\^e}ncia Artificial (APPIA)}
}
""",
    academic_reference_bibtex_key="cortez2007data",
    license="CC BY 4.0",
    data_tags=["IID", "Spatial", "ForcedIIDFromTemporal"],
    curation_comments="""
We start with the data from UCI.

- The original paper used a IID split. The tasks sounds like it requires a temporal split. However, the data does not contain the original time indices (or order of samples) and just month/day over 3 years. Furthermore, the authors made sure to only use time-invariant features as it seems they wanted to develop a time-invaraint model for deployment and analysis of the data. In other words, this task can be handled by IID splits. Furthermore, the task in the paper was about using algorithms also for feature selection (so scientific discovery).
- We log1p scale the target as suggest by the authors.
- The data contains naturally occurring duplicates.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="area",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "forestfires.csv")
print("Loaded data shape:", df.shape)

df["area"] = np.log1p(df["area"])

as_cat_type = ["month", "day"]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (517, 13)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 517
Columns: 13
Use sampling: False (sample size: 517)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['DC', 'DMC', 'temp', 'ISI', 'FFMC', 'RH', 'wind', 'month', 'X', 'day']
Rows remaining as candidates after top-10 filter: 27 (of 517)

#### Duplicate Report
Total duplicate rows: 4 (0.77% of dataset)
Duplicate rows ignoring target: 13 (2.51% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,6,5,may,sat,85.1,28.0,113.8,3.5,11.3,94,4.9,0.0,0.000000
1,7,5,aug,tue,96.1,181.1,671.2,14.3,21.6,65,4.9,0.8,0.000000
2,8,6,aug,mon,92.1,207.0,672.6,8.2,25.5,29,1.8,0.0,0.802002
3,5,4,sep,fri,94.3,85.1,692.3,15.9,20.1,47,4.9,0.0,0.900161
4,2,4,aug,wed,94.5,139.4,689.1,20.0,29.2,30,4.9,0.0,1.081805


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,month,category,0.0,0.0,12.0,"aug, sep, mar, jul, feb, jun, oct, apr, dec, jan"
1,day,category,0.0,0.0,7.0,"sun, fri, sat, mon, tue, thu, wed"
2,FFMC,float64,0.0,0.0,106.0,"91.6, 92.1, 91.0, 91.7, 92.4, 93.7, 92.5, 94.8, 92.9, 90.1"
3,DMC,float64,0.0,0.0,215.0,"99.0, 129.5, 142.4, 231.1, 126.5, 108.3, 137.0, 35.8, 108.4, 25.4"
4,DC,float64,0.0,0.0,219.0,"745.3, 692.6, 692.3, 715.1, 601.4, 698.6, 706.4, 647.1, 80.8, 686.5"
5,ISI,float64,0.0,0.0,119.0,"9.6, 7.1, 6.3, 7.0, 8.4, 6.2, 9.2, 7.5, 8.1, 9.0"
6,temp,float64,0.0,0.0,192.0,"19.6, 17.4, 20.6, 15.4, 19.1, 21.9, 21.6, 20.8, 18.9, 23.4"
7,wind,float64,0.0,0.0,21.0,"3.1, 2.2, 4.0, 4.9, 2.7, 5.4, 4.5, 3.6, 1.8, 5.8"
8,rain,float64,0.0,0.0,7.0,"0.0, 0.8, 0.2, 1.4, 0.4, 6.4, 1.0"
9,area,float64,0.0,0.0,251.0,"0.0, 1.0784, 0.9002, 1.1569, 3.3898, 1.0818, 1.5497, 2.0055, 2.3292, 2.3943"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
X,517.0,4.669246,2.313778,1.0,9.00000
Y,517.0,4.299807,1.229900,2.0,9.00000
FFMC,517.0,90.644681,5.520111,18.7,96.20000
DMC,517.0,110.872340,64.046482,1.1,291.30000
DC,517.0,547.940039,248.066192,7.9,860.60000
ISI,517.0,9.021663,4.559477,0.0,56.10000
temp,517.0,18.889168,5.806625,2.2,33.30000
RH,517.0,44.288201,16.317469,15.0,100.00000
wind,517.0,4.017602,1.791653,0.4,9.40000
rain,517.0,0.021663,0.295959,0.0,6.40000


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column rank                    
day    1      sun     95  18.38
       2      fri     85  16.44
       3      sat     84  16.25
       4      mon     74  14.31
       5      tue     64  12.38
month  1      aug    184  35.59
       2      sep    172  33.27
       3      mar     54  10.44
       4      jul     32   6.19
       5      feb     20   3.87

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,47.78,1.218,0.507,1.956,0.365,log1p,950.1,1542.7,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to forest_fires/019d9cdb-6862-7bcb-8733-cb9db5f573ed
019d9cdb-6862-7bcb-8733-cb9db5f573ed
5121ba3dce8e607823b5b703a7f89683ddd794fb733d812bf869f6d4b15e5b5e
